# Physics-aware vs statistical anomaly detection on the PV dataset
Run top to bottom (Run → Run All Cells). Detector inputs: power P, irradiance G, temperature T. `modeled_power` is the physics reference used only by the physics check/projection.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)

# >>> change this if your CSVs are somewhere else <<<
DATA_DIR = r"C:\Users\Shreya\Downloads"
TRAIN_CSV = os.path.join(DATA_DIR, "pv_train_merged.csv")
TEST_CSV  = os.path.join(DATA_DIR, "pv_test_merged.csv")

## Data, windows, physical limits

In [ ]:
COLS = ["instantaneous_output_power", "instantaneous_global_radiation", "clearsky_Tamb"]  # P, G, T
REF  = "modeled_power"          # physics-model reference power (NOT fed to the detector)
FEATS = ["P", "G", "T"]
L, S = 24, 6                    # window length, stride (rows)

def make_windows(path):
    d = pd.read_csv(path, usecols=COLS + [REF, "label", "label_name", "site_num", "date", "source_file"])
    X, M, lab, alt, site, name = [], [], [], [], [], []
    for _, g in d.groupby(["source_file", "site_num", "date"], sort=False):   # never mix files/days in a window
        a = g[COLS].to_numpy(np.float32); m = g[REF].to_numpy(np.float32)
        altrow = np.abs(a[:, 0] - m) > 1e-6          # row where P differs from the clean reference
        for i in range(0, len(g) - L + 1, S):
            X.append(a[i:i+L]); M.append(m[i:i+L]); lab.append(g.label.iloc[0])
            alt.append(altrow[i:i+L].mean()); site.append(g.site_num.iloc[0]); name.append(g.label_name.iloc[0])
    return dict(X=np.stack(X), M=np.stack(M), lab=np.array(lab), alt=np.array(alt),
                site=np.array(site), name=np.array(name))

tr, te = make_windows(TRAIN_CSV), make_windows(TEST_CSV)

# Anomaly files label EVERY row with the fault type, but only some rows are really altered.
# Keep normal windows, and anomalous windows with at least one altered row. Drop the ambiguous rest.
for d in (tr, te):
    d["keep"] = (d["lab"] == 0) | (d["alt"] > 0)
    d["y"] = (d["lab"] != 0).astype(int)
print("train windows:", len(tr["y"]), "| test windows:", len(te["y"]))
print(pd.Series(te["name"][te["keep"]]).value_counts().to_string())

# train detector on NORMAL windows only; hold out 3 sites for the threshold
nm = tr["lab"] == 0
sites = np.unique(tr["site"][nm]); val_sites = sites[-3:]
is_val = np.isin(tr["site"], val_sites)
Xfit = tr["X"][nm & ~is_val]; Xval = tr["X"][nm & is_val]
print("fit windows:", len(Xfit), "| val windows:", len(Xval))

# min-max scaling to [0,1] (fit on normal training rows)
rows = Xfit.reshape(-1, 3); span = rows.max(0) - rows.min(0)
LO, HI = rows.min(0) - .05 * span, rows.max(0) + .05 * span
to_norm = lambda x: np.clip((x - LO) / (HI - LO), 0, 1).astype(np.float32)
to_raw  = lambda xn: xn * (HI - LO) + LO

# physical limits learned from NORMAL training data
Mfit = tr["M"][nm & ~is_val]
TAU  = np.percentile(np.abs(Xfit[..., 0] - Mfit), 99.9)                  # allowed |P - physics model|
RAMP = np.percentile(np.abs(np.diff(Xfit, axis=1)).reshape(-1, 3), 99.9, axis=0)   # allowed step change per feature
print("TAU =", round(float(TAU), 3), "| RAMP (P,G,T) =", RAMP.round(3))

def project(xn, M, rounds=5):
    """Project windows onto the physical set: |P-M|<=TAU and per-step ramp limits."""
    x = to_raw(xn).copy()
    for _ in range(rounds):
        x[..., 0] = np.clip(x[..., 0], M - TAU, M + TAU)
        for t in range(1, L):
            x[:, t] = np.clip(x[:, t], x[:, t-1] - RAMP, x[:, t-1] + RAMP)
    x[..., 0] = np.clip(x[..., 0], M - TAU, M + TAU)
    return to_norm(x)

def physical_ok(xn, M, slack=1.05):
    """True for windows that satisfy every physical rule (this is the 'physics check' defender)."""
    x = to_raw(xn)
    c1 = (np.abs(x[..., 0] - M) <= TAU * slack + 1e-3).all(1)
    c2 = (np.abs(np.diff(x, axis=1)) <= RAMP * slack + 1e-3).all((1, 2))
    return c1 & c2

## Train the LSTM autoencoder (normal data only)

In [ ]:
class AE(nn.Module):
    def __init__(s, f=3, h=32):
        super().__init__()
        s.enc = nn.LSTM(f, h, batch_first=True); s.dec = nn.LSTM(h, h, batch_first=True); s.out = nn.Linear(h, f)
    def forward(s, x):
        _, (h, _) = s.enc(x)
        z = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        return s.out(s.dec(z)[0])

def err(model, x):
    with torch.no_grad():
        x = torch.as_tensor(x); return ((model(x) - x) ** 2).mean((1, 2)).numpy()

model = AE(); opt = torch.optim.Adam(model.parameters(), 1e-3)
trn = torch.as_tensor(to_norm(Xfit)); losses = []
for ep in range(30):
    tot = 0
    for b in torch.randperm(len(trn)).split(64):
        loss = ((model(trn[b]) - trn[b]) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    losses.append(tot / len(trn))
    if ep % 5 == 0: print("epoch", ep, "loss", round(losses[-1], 5))
model.eval()
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("train MSE"); plt.title("Training loss"); plt.show()

## Test on clean data: statistical vs physics detection

In [ ]:
thr = np.percentile(err(model, to_norm(Xval)), 99)     # alarm threshold from held-out normal sites
k = te["keep"]
Xt, Mt, yt, nt = to_norm(te["X"][k]), te["M"][k], te["y"][k], te["name"][k]
e = err(model, Xt); flag_ae = e > thr
flag_phys = ~physical_ok(Xt, Mt)
print(f"LSTM-AE   : detects {flag_ae[yt==1].mean():.2f} of faults | false alarms {flag_ae[yt==0].mean():.2f}")
print(f"Physics   : detects {flag_phys[yt==1].mean():.2f} of faults | false alarms {flag_phys[yt==0].mean():.2f}")
tab = pd.DataFrame({"type": nt, "LSTM-AE": flag_ae, "Physics check": flag_phys})
tab = tab[yt == 1].groupby("type").mean().round(2); print(tab)
tab.plot.bar(figsize=(8, 3.5)); plt.ylabel("detection rate"); plt.title("Detection rate by fault type (clean data)"); plt.tight_layout(); plt.show()
plt.hist(e[yt==0], bins=40, alpha=.6, label="normal"); plt.hist(e[yt==1], bins=40, alpha=.6, label="faulty")
plt.axvline(thr, color="k", ls="--", label="threshold"); plt.xscale("log"); plt.legend(); plt.xlabel("reconstruction error"); plt.show()

## Adversarial attacks on the faulty test windows

In [ ]:
def attack(model, xn, M, eps, steps, alpha, physics):
    x0 = torch.as_tensor(xn); x = x0.clone()
    for _ in range(steps):
        x.requires_grad_(True)
        loss = ((model(x) - x) ** 2).mean((1, 2)).sum()
        g, = torch.autograd.grad(loss, x)
        x = x.detach() - alpha * g.sign()                     # push reconstruction error DOWN
        x = torch.min(torch.max(x, x0 - eps), x0 + eps).clamp(0, 1)
        if physics: x = torch.as_tensor(project(x.numpy(), M))
    return x.detach().numpy()

rs = np.random.RandomState(0)
idx = rs.choice(np.where(yt == 1)[0], size=min(500, (yt == 1).sum()), replace=False)
A, MA = Xt[idx], Mt[idx]
print("attacking", len(A), "faulty windows")
res, epss = {}, [0.02, 0.05, 0.1, 0.2]
for eps in epss:
    for name, steps, phys in [("FGSM", 1, False), ("PGD", 20, False), ("PGD+physics", 20, True)]:
        adv = attack(model, A, MA, eps, steps, eps if steps == 1 else eps / 5, phys)
        evade = (err(model, adv) < thr).mean(); ok = physical_ok(adv, MA).mean()
        size = np.abs(adv - A).max((1, 2)).mean()
        res[(name, eps)] = (evade, ok, adv)
        print(f"eps={eps:<5} {name:<12} evasion={evade:.2f} phys-valid={ok:.2f} avg-perturbation={size:.2f}")

## Results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in ["FGSM", "PGD", "PGD+physics"]:
    ax[0].plot(epss, [res[(n, e)][0] for e in epss], "o-", label=n)
    ax[1].plot(epss, [res[(n, e)][1] for e in epss], "o-", label=n)
ax[0].set_title("Evasion rate (attacker wins)"); ax[1].set_title("Fraction physically valid")
for a in ax: a.set_xlabel("epsilon"); a.set_ylim(-.05, 1.05); a.legend()
plt.tight_layout(); plt.show()

eps, i = 0.1, 0
fig, axs = plt.subplots(1, 3, figsize=(15, 3.2))
for k, a in enumerate(axs):
    a.plot(to_raw(A[i])[:, k], "k", lw=2, label="faulty (original)")
    if k == 0: a.plot(MA[i], "g--", label="physics model"); a.fill_between(range(L), MA[i]-TAU, MA[i]+TAU, color="g", alpha=.1)
    for n in ["FGSM", "PGD", "PGD+physics"]: a.plot(to_raw(res[(n, eps)][2][i])[:, k], label=n)
    a.set_title(FEATS[k])
axs[0].legend(fontsize=7); plt.tight_layout(); plt.show()